### 1.2 COMPARATIVE ANALYSIS: NeuroKitDelineator vs WTdelineatorWrapper

In [1]:
import wfdb
import matplotlib.pyplot as plt
from signal_utils import expand_to_12_leads

# Load the signal from data/staff_III/001c
record = wfdb.rdrecord('data/staff_III/data/test')
staff_signal = record.p_signal[:5000].T # type: ignore
staff_signal, signal_12ch_names = expand_to_12_leads(staff_signal)

staff_sampling_rate = record.fs
staff_channel_names = signal_12ch_names

print(f"Signal shape: {staff_signal.shape}")
print(f"Sampling rate: {staff_sampling_rate} Hz")
print(f"Number of channels: {staff_signal.shape[0]}")
print(f"Channel names: {staff_channel_names}")


Signal shape: (12, 5000)
Sampling rate: 1000 Hz
Number of channels: 12
Channel names: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'I', 'II', 'III', 'aVR', 'aVL', 'aVF']


In [3]:
import plotly.graph_objects as go
import plotly.subplots as sp
from typing import List, Dict
import numpy as np


class DebugVisualizer:
    """Interactive visualization of ECG signals with delineation points using Plotly"""
    
    # Define visual properties for each peak type
    # Organized by wave (P, Q, R, S, T) with different symbols for onsets, peaks, offsets
    PEAK_STYLES = {
        # P wave - Blue family
        'P_onsets': {'color': 'blue', 'symbol': 'triangle-up', 'size': 9},
        'P_peaks': {'color': 'blue', 'symbol': 'circle', 'size': 9},
        'P2_peaks': {'color': 'blue', 'symbol': 'square', 'size': 9},
        'P_offsets': {'color': 'blue', 'symbol': 'triangle-down', 'size': 9},
        
        # Q wave - Green (only peak)
        'Q_peaks': {'color': 'green', 'symbol': 'circle', 'size': 9},
        
        # R wave - Red family
        'R_onsets': {'color': 'red', 'symbol': 'triangle-up', 'size': 9},
        'R_peaks': {'color': 'red', 'symbol': 'circle', 'size': 9},
        'R_offsets': {'color': 'red', 'symbol': 'triangle-down', 'size': 9},
        
        # S wave - Purple (only peak)
        'S_peaks': {'color': 'mediumpurple', 'symbol': 'circle', 'size': 9},
        
        # T wave - Orange family
        'T_onsets': {'color': 'orange', 'symbol': 'triangle-up', 'size': 9},
        'T_peaks': {'color': 'orange', 'symbol': 'circle', 'size': 9},
        'T2_peaks': {'color': 'orange', 'symbol': 'square', 'size': 9},
        'T_offsets': {'color': 'orange', 'symbol': 'triangle-down', 'size': 9},
    }
    
    def __init__(self, signal: np.ndarray, qrs_features: List[Dict[str, list]], 
                 fs: int, channel_names: List[str] = None):
        """
        Initialize the visualizer
        
        Args:
            signal: (n_channels, n_samples) signal array
            qrs_features: List of dicts, one per channel with peak/offset info
            fs: Sampling rate
            channel_names: Optional list of channel names
        """
        self.signal = signal
        self.qrs_features = qrs_features
        self.fs = fs
        self.n_channels = signal.shape[0]
        self.n_samples = signal.shape[1]
        self.time = np.arange(self.n_samples) / fs  # Convert to seconds
        
        if channel_names is None:
            self.channel_names = [f"Channel {i}" for i in range(self.n_channels)]
        else:
            self.channel_names = channel_names
    
    def _convert_indices_to_time(self, indices: np.ndarray) -> np.ndarray:
        """Convert sample indices to time in seconds"""
        indices_clean = indices[~np.isnan(indices)].astype(int)
        return self.time[indices_clean]
    
    def _get_signal_values(self, indices: np.ndarray, channel_idx: int) -> np.ndarray:
        """Get signal values at given indices"""
        indices_clean = indices[~np.isnan(indices)].astype(int)
        return self.signal[channel_idx, indices_clean]
    
    def plot(self, channel_indices: List[int] = None, height_per_channel: int = 300) -> go.Figure:
        """
        Create interactive plot
        
        Args:
            channel_indices: List of channel indices to plot (default: all)
            height_per_channel: Height in pixels for each subplot
        
        Returns:
            Plotly figure object
        """
        if channel_indices is None:
            channel_indices = list(range(self.n_channels))
        
        # Validate channel indices
        channel_indices = [i for i in channel_indices if 0 <= i < self.n_channels]
        
        n_plots = len(channel_indices)
        height = height_per_channel * n_plots + 100
        
        # Create subplots
        fig = sp.make_subplots(
            rows=n_plots, 
            cols=1,
            subplot_titles=tuple(self.channel_names[i] for i in channel_indices),
            shared_xaxes=True,
            vertical_spacing=0.08
        )
        
        # Add traces for each channel
        for plot_idx, ch_idx in enumerate(channel_indices):
            row = plot_idx + 1
            
            # Add the signal trace
            fig.add_trace(
                go.Scatter(
                    x=self.time,
                    y=self.signal[ch_idx, :],
                    mode='lines',
                    name=f'{self.channel_names[ch_idx]} (signal)',
                    line=dict(color='steelblue', width=1),
                    hovertemplate='<b>Signal</b><br>Time: %{x:.3f}s<br>Amplitude: %{y:.3f}<extra></extra>',
                ),
                row=row, col=1
            )
            
            # Add peaks and offsets for this channel
            if ch_idx < len(self.qrs_features):
                features = self.qrs_features[ch_idx]
                
                for peak_key, peak_indices in features.items():
                    # Skip RR which is a single value, not indices
                    if peak_key == 'RR':
                        continue
                    
                    if peak_indices is None or (isinstance(peak_indices, float) and np.isnan(peak_indices)):
                        continue
                    
                    # Handle case where peak_indices might be a scalar
                    if isinstance(peak_indices, (int, float, np.integer)):
                        peak_indices = np.array([peak_indices])
                    else:
                        peak_indices = np.array(peak_indices)
                    
                    if len(peak_indices) == 0:
                        continue
                    
                    # Get style for this peak type
                    style = self.PEAK_STYLES.get(peak_key, {
                        'color': 'gray', 'symbol': 'circle', 'size': 8
                    })
                    
                    # Convert indices to time and get values
                    peak_times = self._convert_indices_to_time(peak_indices)
                    peak_values = self._get_signal_values(peak_indices, ch_idx)
                    
                    if len(peak_times) > 0:
                        fig.add_trace(
                            go.Scatter(
                                x=peak_times,
                                y=peak_values,
                                mode='markers',
                                name=peak_key,
                                marker=dict(
                                    color=style['color'],
                                    symbol=style['symbol'],
                                    size=style['size'],
                                    line=dict(color='white', width=1)
                                ),
                                hovertemplate=f'<b>{peak_key}</b><br>Time: %{{x:.3f}}s<br>Amplitude: %{{y:.3f}}<extra></extra>',
                            ),
                            row=row, col=1
                        )
        
        # Update layout
        fig.update_layout(
            title_text="ECG Signal with Delineation Points",
            height=height,
            hovermode='x unified',
            font=dict(size=11),
        )
        
        # Update x-axis label
        fig.update_xaxes(title_text="Time (s)", row=n_plots, col=1)
        
        # Update y-axis labels
        for i, ch_idx in enumerate(channel_indices):
            fig.update_yaxes(title_text="Amplitude (mV)", row=i+1, col=1)
        
        return fig
    
    def show(self, channel_indices: List[int] = None):
        """Display the plot"""
        fig = self.plot(channel_indices)
        fig.show()


In [ ]:
from delineators import NeuroKitDelineator

nk_delineator = NeuroKitDelineator()
cleaned_signal, QRS_features = nk_delineator.delineate(staff_signal, staff_sampling_rate)
display(QRS_features[2])
visualizer = DebugVisualizer(cleaned_signal, QRS_features, staff_sampling_rate, staff_channel_names)
visualizer.show(channel_indices=[2])

{'R_peaks': array([1057, 1952, 2840, 3734, 4613]),
 'P_peaks': array([ 966, 1861, 2749, 3643, 4485]),
 'Q_peaks': array([1010, 1899, 2795, 3682, 4559]),
 'S_peaks': array([1087, 1982, 2870, 3764, 4644]),
 'T_peaks': array([1336, 2236, 3123, 4015, 4897]),
 'P_onsets': array([ 850, 1782, 2671, 3570, 4443]),
 'P_offsets': array([ 878, 1836, 2724, 3623, 4506]),
 'R_onsets': array([ 970, 1865, 2753, 3648, 4528]),
 'R_offsets': array([1113, 2009, 2897, 3791, 4671]),
 'T_onsets': array([1255, 2152, 3043, 3938, 4811]),
 'T_offsets': array([1441, 2339, 3225, 4121, 4983]),
 'RR': 889}

In [5]:
from delineators import WTdelineatorWrapper

WT_delineator = WTdelineatorWrapper()
cleaned_signal, QRS_features = WT_delineator.delineate(staff_signal, staff_sampling_rate)
print(staff_channel_names)
display(QRS_features[2])
visualizer = DebugVisualizer(cleaned_signal, QRS_features, staff_sampling_rate, staff_channel_names)
visualizer.show(channel_indices=[2])


['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'I', 'II', 'III', 'aVR', 'aVL', 'aVF']


{'P_peaks': array([  nan,  912., 1820., 2692., 3544., 4469.]),
 'P2_peaks': array([  nan,  993., 1886., 2777., 3597., 4550.]),
 'Q_peaks': array([nan, nan, nan, nan, nan, nan]),
 'R_peaks': array([ 171., 1055., 1950., 2838., 3732., 4611.]),
 'S_peaks': array([ 202., 1085., 1980., 2868., 3762., 4642.]),
 'T_peaks': array([ 441., 1325., 2221., 3109., 4002.,   nan]),
 'T2_peaks': array([  nan,   nan,   nan, 3272., 4163.,   nan]),
 'P_onsets': array([  nan,  849., 1745., 2632., 3458., 4410.]),
 'P_offsets': array([  nan, 1003., 1898., 2787., 3633., 4559.]),
 'R_onsets': array([ 129., 1012., 1906., 2795., 3689., 4570.]),
 'R_offsets': array([ 237., 1120., 2016., 2904., 3798., 4677.]),
 'T_onsets': array([ 313., 1192., 2091., 2983., 3877.,   nan]),
 'T_offsets': array([ 540., 1421., 2320., 3345., 4242.,   nan])}

In [8]:
# =====================================================================
# COMPARATIVE ANALYSIS: NeuroKitDelineator vs WTdelineatorWrapper
# =====================================================================
from collections import defaultdict
import warnings
import os
import glob
import pandas as pd

def count_detections(ecg_parameters: list[Dict[str, np.ndarray]]) -> Dict[int, Dict[str, int]]:
    """
    Count how many detections were made for each property in each channel.
    Handles NaN values and various array formats safely.
    
    Args:
        ecg_parameters: List of dicts (one per channel) with delineation results
    
    Returns:
        Dict mapping channel_idx -> {property: count_of_detections}
    """
    detection_counts = {}
    
    for ch_idx, channel_params in enumerate(ecg_parameters):
        detection_counts[ch_idx] = {}
        
        for prop_name, values in channel_params.items():
            if prop_name == 'RR':  # Skip scalar values
                continue
            
            try:
                if isinstance(values, np.ndarray):
                    # Count finite (non-NaN, non-inf) values
                    # np.isfinite handles both NaN and inf, returns boolean array
                    count = int(np.count_nonzero(np.isfinite(values)))
                elif isinstance(values, (list, tuple)):
                    # Handle list/tuple - convert to array first
                    arr = np.asarray(values, dtype=float)
                    count = int(np.count_nonzero(np.isfinite(arr)))
                elif np.isnan(values):
                    # Single NaN value
                    count = 0
                elif np.isfinite(values):
                    # Single finite value
                    count = 1
                else:
                    count = 0
            except (TypeError, ValueError):
                # Catch any conversion errors
                count = 0
            
            detection_counts[ch_idx][prop_name] = count
    
    return detection_counts


def compare_delineators_on_dataset(data_dir: str = "data/staff_III/data", max_records: int = None, channel_names: list = None):
    """
    Compare NeuroKitDelineator and WTdelineatorWrapper across local dataset.
    
    Args:
        data_dir: Local directory containing .hea and .dat files
        max_records: Max number of records to process (None = all)
        channel_names: Optional list of channel names for output
    
    Returns:
        Tuple of (summary_df, detailed_results)
    """
    # Find all .hea files in the directory
    hea_files = sorted(glob.glob(f"{data_dir}/*.hea"))
    record_names = [os.path.basename(f)[:-4] for f in hea_files]  # Remove .hea extension
    
    if max_records:
        record_names = record_names[:max_records]
    
    print(f"Found {len(hea_files)} total .hea files in {data_dir}")
    print(f"Processing {len(record_names)} records...\n")
    
    # Initialize aggregators: channel_idx -> property -> list of counts across records
    nk_results = defaultdict(lambda: defaultdict(list))
    wt_results = defaultdict(lambda: defaultdict(list))
    
    failed_records = []
    processed_count = 0
    
    for record_idx, record_name in enumerate(record_names):
        try:
            # Load record from local path
            record_path = os.path.join(data_dir, record_name)
            record = wfdb.rdrecord(record_path)
            signal = record.p_signal[:5000].T  # (n_channels, 5000)
            fs = record.fs
            
            # Run both delineators
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                try:
                    nk_delineator = NeuroKitDelineator()
                    nk_cleaned, nk_params = nk_delineator.delineate(signal, fs)
                    nk_counts = count_detections(nk_params)
                except Exception as e_nk:
                    print(f"  ⚠ NK error on {record_name}: {str(e_nk)[:60]}")
                    nk_counts = {}
                
                try:
                    wt_delineator = WTdelineatorWrapper()
                    wt_cleaned, wt_params = wt_delineator.delineate(signal, fs)
                    wt_counts = count_detections(wt_params)
                except Exception as e_wt:
                    print(f"  ⚠ WT error on {record_name}: {str(e_wt)[:60]}")
                    wt_counts = {}
            
            # Aggregate results - only add if we got valid data
            if nk_counts:
                for ch_idx, props_dict in nk_counts.items():
                    for prop_name, count in props_dict.items():
                        if isinstance(count, (int, np.integer)):
                            nk_results[ch_idx][prop_name].append(count)
            
            if wt_counts:
                for ch_idx, props_dict in wt_counts.items():
                    for prop_name, count in props_dict.items():
                        if isinstance(count, (int, np.integer)):
                            wt_results[ch_idx][prop_name].append(count)
            
            processed_count += 1
            if (processed_count % max(1, len(record_names)//10) == 0):
                print(f"  Processed {processed_count}/{len(record_names)} records...")
        
        except Exception as e:
            failed_records.append((record_name, str(e)))
            if len(failed_records) <= 3:
                print(f"  ❌ Error on {record_name}: {str(e)[:80]}")
    
    print(f"\n✓ Processed {processed_count} records successfully ({len(failed_records)} failed)\n")
    
    # Build comparison summary
    summary_data = []
    
    # Get all properties from both delineators
    all_properties = set()
    for ch_results in nk_results.values():
        all_properties.update(ch_results.keys())
    for ch_results in wt_results.values():
        all_properties.update(ch_results.keys())
    
    # Aggregate across all records for each property and channel
    for prop_name in sorted(all_properties):
        for ch_idx in range(12):  # 12 channels from 9 + 3 derived
            nk_total = sum(nk_results[ch_idx][prop_name]) if ch_idx in nk_results and prop_name in nk_results[ch_idx] else 0
            wt_total = sum(wt_results[ch_idx][prop_name]) if ch_idx in wt_results and prop_name in wt_results[ch_idx] else 0
            
            # Only include if at least one detected something
            if nk_total > 0 or wt_total > 0:
                summary_data.append({
                    'property': prop_name,
                    'channel': ch_idx,
                    'NeuroKit_detections': nk_total,
                    'WT_detections': wt_total,
                    'difference': wt_total - nk_total,
                    'winner': 'WT' if wt_total > nk_total else ('NK' if nk_total > wt_total else 'TIE')
                })
    
    if not summary_data:
        print("⚠ No detections found! Check that records loaded correctly.")
        return pd.DataFrame(), (nk_results, wt_results, processed_count)
    
    summary_df = pd.DataFrame(summary_data)
    # Map channel index to channel name for each row (FIX: don't assign list directly)
    if channel_names:
        summary_df['channel_name'] = summary_df['channel'].apply(
            lambda x: channel_names[x] if x < len(channel_names) else f"Ch{x}"
        )
    return summary_df, (nk_results, wt_results, processed_count)


# Run the comparison
print("Starting comparative analysis of delineators...\n")
summary_df, details = compare_delineators_on_dataset(
    data_dir="data/staff_III/data", 
    max_records=100,
    channel_names=staff_channel_names
)

if not summary_df.empty:
    # Summary by property
    print("\n" + "="*80)
    print("SUMMARY BY PROPERTY (across all channels)")
    print("="*80)
    property_agg = summary_df.groupby('property').agg({
        'NeuroKit_detections': 'sum',
        'WT_detections': 'sum',
    }).reset_index()
    property_agg['difference'] = property_agg['WT_detections'] - property_agg['NeuroKit_detections']
    property_agg['winner'] = property_agg['difference'].apply(
        lambda x: 'WT' if x > 0 else ('NK' if x < 0 else 'TIE')
    )
    print(property_agg.to_string(index=False))

    # Summary by channel
    print("\n" + "="*80)
    print("SUMMARY BY CHANNEL (across all properties)")
    print("="*80)
    channel_agg = summary_df.groupby('channel').agg({
        'NeuroKit_detections': 'sum',
        'WT_detections': 'sum',
    }).reset_index()
    channel_agg['difference'] = channel_agg['WT_detections'] - channel_agg['NeuroKit_detections']
    channel_agg['winner'] = channel_agg['difference'].apply(
        lambda x: 'WT' if x > 0 else ('NK' if x < 0 else 'TIE')
    )
    # Map channel indices to channel names
    channel_agg['channel_name'] = channel_agg['channel'].apply(lambda x: staff_channel_names[x] if x < len(staff_channel_names) else f"Ch{x}")
    # Reorder columns to show channel name first
    channel_agg = channel_agg[['channel', 'channel_name', 'NeuroKit_detections', 'WT_detections', 'difference', 'winner']]
    print(channel_agg.to_string(index=False))
else:
    print("Summary DataFrame is empty. Check data_dir path and record loading.")

Starting comparative analysis of delineators...

Found 521 total .hea files in data/staff_III/data
Processing 100 records...

  ⚠ NK error on 002a: cannot convert float NaN to integer
  Processed 10/100 records...
  Processed 20/100 records...
  Processed 30/100 records...
  Processed 40/100 records...
  Processed 50/100 records...
  ⚠ NK error on 013a: cannot convert float NaN to integer
  Processed 60/100 records...
  ⚠ NK error on 013e: cannot convert float NaN to integer
  Processed 70/100 records...
  Processed 80/100 records...
  ⚠ NK error on 016f: integer division or modulo by zero
  Processed 90/100 records...
  Processed 100/100 records...

✓ Processed 100 records successfully (0 failed)


SUMMARY BY PROPERTY (across all channels)
 property  NeuroKit_detections  WT_detections  difference winner
 P2_peaks                    0           3965        3965     WT
P_offsets                 4796           4771         -25     NK
 P_onsets                 4740           4769         